# Rung 3 — GPU page decode (the emitter's heavy half, offloaded)

Slices + decodes all **1,412 matched real pages** (both sources) with `rung3-labeler` on the GPU,
writing the `<page>_decode.json` cache files the laptop emitter reuses — so the laptop only runs
the cool, cheap alignment afterwards. ~30–60 min on a T4, faster on L4/A100.

**Before running:** upload `data/colab/tnc_rung3_decode_colab.zip` (1.1 GB) to `MyDrive/tnc/`.
The `rung3-labeler` weights are already on Drive from the fine-tune (`tnc/rung3-labeler/best`).

In [ ]:
# Which GPU did we get?
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy the data package Drive -> VM disk and unzip (fast local disk).
%%time
!cp /content/drive/MyDrive/tnc/tnc_rung3_decode_colab.zip /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/tnc_rung3_decode_colab.zip
!wc -l /content/tnc/data/colab/decode_pages.txt

In [ ]:
# Dependencies (torch is preinstalled).
!pip -q install transformers opencv-python-headless

In [ ]:
# Labeler weights Drive -> VM disk (skip the 1.1 GB trainer_state.pt).
%%time
!mkdir -p /content/rung3-labeler
!rsync -a --exclude trainer_state.pt /content/drive/MyDrive/tnc/rung3-labeler/best/ /content/rung3-labeler/
!ls /content/rung3-labeler

In [ ]:
# SMOKE (~1 min): first 5 pages. Expect "device=cuda", strips sliced, 5/5 done.
%cd /content/tnc
!head -5 data/colab/decode_pages.txt > /content/smoke_pages.txt
!python scripts/rung3/decode_pages_gpu.py --pages /content/smoke_pages.txt \
    --checkpoint /content/rung3-labeler --out data/real/strips --batch-size 32

In [ ]:
# FULL RUN: all pages. --skip-existing makes re-runs resume after a disconnect.
%cd /content/tnc
!python scripts/rung3/decode_pages_gpu.py --pages data/colab/decode_pages.txt \
    --checkpoint /content/rung3-labeler --out data/real/strips --batch-size 32 --skip-existing

In [ ]:
# Package the strips + decode JSONs back to Drive (~1 GB).
%%time
%cd /content/tnc
!zip -1 -q -r /content/drive/MyDrive/tnc/rung3_strips_decoded.zip data/real/strips
!ls -lh /content/drive/MyDrive/tnc/rung3_strips_decoded.zip

## After the run

1. Download `MyDrive/tnc/rung3_strips_decoded.zip` to the repo root.
2. Unpack it INTO the repo (it merges into `data/real/strips/`):
   `unzip -o rung3_strips_decoded.zip`
3. Tell Claude — the emitter then runs locally in minutes using these cached decodes
   (no hot Mac): calibration report → two-source exam freeze → training + exam emits.